In [1]:
import sys
import os

sys.path.append(os.path.abspath("../../"))


In [2]:
!ls

__init__.py       db                model_design_1.py states.py
__pycache__       graph.py          prompt            test.ipynb


In [3]:
from __future__ import annotations

from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, RemoveMessage

from graph import LangGraph
from dotenv import load_dotenv


In [4]:
load_dotenv()


True

In [5]:
memory = MemorySaver()
langgraph = LangGraph()
app = langgraph.build_graph(checkpointer=memory)

thread_id = "test_user_01"
config = {"configurable": {"thread_id": thread_id}}

[RAG-BUILDER] loading DB with path: ../../db/chroma_db
[RETRIEVER] making dense retriever with TYPE: similarity and ARGS: {'k': 20}
[RETRIEVER] making bm25 retriever with PATH: ../../db/bm25


In [6]:
# LLM 초기화 및 함수 호출을 통한 구조화된 출력 생성
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from states import State, RouterDecision
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model(
            "gpt-4o-mini",
            # model_provider="upstage",
            temperature=0,
        )

structured_llm = llm.with_structured_output(RouterDecision)

# 시스템 메시지와 사용자 질문을 포함한 프롬프트 템플릿 생성
ROUTER_SYSTEM_PROMPT = "hi"


In [7]:

from langchain_core.prompts import load_prompt
from datetime import datetime
from src.graph.states import RagDocument
import yaml
from prompt.prompt import Router, ChatBot, Summarizer, Validator
# 1. router 용
days = ["월", "화", "수", "목", "금", "토", "일"]
now = datetime.now()
day_of_week = days[now.weekday()]
current_time = f"{now.strftime('%Y년 %m월 %d일')} {day_of_week}요일"

# 2. validator 용
documents = [RagDocument(text="장소명: 원조전복죽집\n구분: 음식점\n설명: 게우내장가 있는 전복을 통째로 끓인, 녹색의오리지널 전복죽을 맛볼 수 있는 곳이다. 참기름과 전복만 사용하는 것이 특징. 전복과 해초, 나물 등이 푸짐히 들어간 비빔밥도 식사메뉴로 인기가 많다. 50년 넘는 역사를 자랑한다.\n메뉴: 로 인기가 많다. 50년 넘는 역사를 자랑한다.메뉴 원조전복죽1만4천원, 특 1만9천원, 프리미엄전복죽소 2만4천원, 대 3만6천원, 전복해초비빔밥1만4천원, 전복회, 전복버터구이각 소 3만7천원, 대 5만8천원, 전복콩나물해장국9천원\n영업시간: 0630 2300  연중무휴\n주소: 부산광역시 해운대구 해운대해변로298번길 24 중동\n연락처: 0517424690\n출처: 블루리본2025'"),
             RagDocument(text="장소명: 히말라야 싱잉볼\n구분: 체험\n설명: 히말라야 싱잉볼 치유명상 한국본부는 히말라야의 자연 에너지와 전통 사운드 테라피를 바탕으로 현대인의 심신 회복과 정서 안정을 돕습니다. 부산의 바다와 히말라야의 울림이 어우러진 이곳은 단순한 명상을 넘어 내면과 마주하는 치유의 여정을 제공합니다. 휴식을 넘어 자신과 깊이 연결되는 특별한 공간입니다.\n주소: 부산시 해운대구 해운대로 723 하모니타워 37층 102동 3712호\n연락처: 01023576262\n출처: 부산럭셔리관광2025"),  
             RagDocument(text="장소명: 탐복\n구분: 음식점\n설명: 고급스러운 분위기의 전복 전문점. 기본 세트를 주문하면 탐복밥, 탐복죽, 탐복구이가 함께 나온다. 탐복밥의 고명은 미역, 톳, 곤드레 중에 고를 수 있으며 게우소스와 전복장으로 비벼 먹는다.\n메뉴: 탐복죽, 탐복밥각 1만7천원, 탐복구이, 탐복청주찜 각 3만8천원\n영업시간: 10301500  토, 일요일 1030150017001930  월, 화요일 휴무\n주소: 부산 기장군 일광읍 문오성길 31\n연락처: 0517274213\n출처: 블루리본2023"),
             RagDocument(text="장소명: 낙동강생태탐방선\n구분: 체험\n설명: 낙동강 에코호라는 이름으로도 불린다. 아름다운 자연 속에 동화되어 보는 시간을 누릴 수 있다.\n주소: 부산 사하구 하단동 114910\n연락처: 0512942135\n출처: Theme2025"),
             RagDocument(text="장소명: 탐복\n구분: 음식점\n설명: 예쁜 플레이팅에 담긴 국내산 전복요리, 시그니처인 전복 내장 소스 등 전복요리를 전문으로 한 캐주얼 한식을 선보이는 곳으로 오래된 가옥을 활용한 한국식캐주얼 인테리어가 돋보인다.\n메뉴: 탐복죽, 탐복밥 17,000, 탐복구이 38,000\n영업시간: 10301600, 월, 화요일 휴무\n주소: 기장군 일광읍 문오성길 31\n연락처: 0517274213\n출처: 부산의 맛2025")
             ]
web_results = [
    '부산은 정말 다양한 매력과 즐길 거리가 많은 도시라, 어떤 스타일의 숙소를 선호하시는지, 어떤 목적으로 방문하시는지에 따라 추천이 달라질 수 있습니다.\n\n**1. 해운대 (Haeundae) - 럭셔리, 해변, 휴양, 가족여행, 커플여행**\n*   **특징:** 부산 대표 해변으로, 고급 호텔, 리조트, 레스토랑, 유흥시설이 밀집해 있습니다. 해변 접근성이 좋고, 밤에도 활기찬 분위기를 즐길 수 있습니다. 동백섬, 아쿠아리움, LCT 랜드마크 등 볼거리도 많습니다.\n*   **추천 숙소:**\n    *   **파라다이스 호텔 부산:** 해운대 대표 럭셔리 호텔. 오션뷰 룸과 야외 스파, 수영장 시설이 매우 좋습니다. 가족, 커플에게 강력 추천.\n    *   **그랜드 조선 부산:** 해운대 해변 바로 앞에 위치한 신상 호텔. 모던하고 세련된 인테리어, 훌륭한 서비스와 부대시설을 자랑합니다.\n    *   **시그니엘 부산:** LCT 랜드마크 타워에 위치한 초고층 럭셔리 호텔. 모든 객실에서 환상적인 오션뷰를 즐길 수 있습니다. 특별한 날을 기념하거나 럭셔리한 휴식을 원할 때 좋습니다.\n    *   **파크 하얏트 부산:** 마린시티에 위치하며, 광안대교와 요트경기장 뷰가 매력적입니다. 해변과는 조금 떨어져 있지만, 조용하고 고급스러운 분위기를 선호하는 분께 좋습니다.\n    *   **신라스테이 해운대 / 라마다 앙코르 바이 윈덤 해운대:** 합리적인 가격에 깔끔하고 편리한 비즈니스 호텔을 찾는 분께 적합합니다. 해변과도 가깝습니다.\n\n**2. 광안리 (Gwanganli) - 야경, 로맨틱, 감성 카페, 산책**\n*   **특징:** 광안대교 야경이 매우 아름다운 해변입니다. 해운대보다는 조금 더 조용하고 로맨틱한 분위기를 느낄 수 있으며, 감성적인 카페와 레스토랑이 많습니다.\n*   **추천 숙소:**\n    *   **켄트 호텔 광안리 by 롯데:** 광안대교 뷰 객실이 유명합니다. 루프탑 바에서 야경을 즐기기 좋습니다.\n    *   **H 에비뉴 호텔 광안리점:** 광안리 해변에 위치한 감성적인 호텔. 깔끔한 인테리어와 오션뷰 객실이 인기입니다.\n    *   **호메르스 호텔:** 광안대교가 한눈에 보이는 오래된 호텔이지만, 접근성 좋고 뷰가 훌륭합니다.\n\n**3. 남포동 / 자갈치 / 국제시장 (Nampo-dong / Jagalchi / Gukje Market) - 관광, 먹거리, 시장 체험, 올드부산**\n*   **특징:** 부산의 옛 정취를 느낄 수 있는 곳. 자갈치시장, 국제시장, BIFF 광장, 용두산공원 등 관광 명소가 밀집해 있습니다. 먹거리가 풍부하고 저렴한 숙소 옵션도 많습니다.\n*   **추천 숙소:**\n    *   **아벤트리 호텔 부산:** 남포동 중심에 위치하여 편리한 접근성을 자랑합니다. 깔끔하고 합리적인 가격.\n    *   **브라운도트 호텔 부산남포:** 주변에 여러 지점이 있으며, 가성비 좋은 깔끔한 부티크 호텔입니다.\n    *   **게스트하우스/호스텔:** 시장 근처에 다양한 저렴한 게스트하우스와 호스텔이 많아 배낭여행객이나 알뜰 여행객에게 좋습니다. (예: 팝콘 호스텔, 이바구 게스트하우스 등)\n\n**4. 서면 (Seomyeon) - 쇼핑, 번화가, 교통의 요지, 비즈니스**\n*   **특징:** 부산의 중심 번화가로, 쇼핑, 맛집, 유흥시설이 밀집해 있습니다. 지하철 1, 2호선이 교차하여 부산 어디든 이동하기 편리합니다. 비즈니스 목적이나 역동적인 도시 분위기를 선호하는 분께 적합합니다.\n*   **추천 숙소:**\n    *   **롯데호텔 부산:** 서면 대표 럭셔리 호텔. 백화점과 연결되어 있어 편리합니다.\n    *   **아르반 호텔:** 깔끔하고 모던한 디자인의 비즈니스 호텔로, 서면역과 가깝습니다.\n    *   **부산 비즈니스 호텔 / 토요코인 부산 서면:** 합리적인 가격의 비즈니스 호텔로, 출장이나 실속 여행에 좋습니다.\n\n**5. 부산역 주변 (Busan Station Area) - 짧은 체류, 편리한 교통**\n*   **특징:** KTX 이용 시 접근성이 가장 좋습니다. 짧은 체류나 다음 날 일찍 이동해야 할 때 편리합니다. 차이나타운, 초량 이바구길 등이 근처에 있습니다.\n*   **추천 숙소:**\n    *   **아스티 호텔 부산:** 부산역 바로 앞에 위치한 깔끔하고 모던한 호텔. 부산항대교 뷰 객실도 있습니다.\n    *   **크라운 하버 호텔 부산:** 부산역과 도보 거리에 있으며, 가격 대비 만족도가 높은 비즈니스 호텔입니다.\n    *   **토요코인 부산역1, 2:** 일본계 비즈니스 호텔로, 깔끔하고 합리적인 가격에 조식도 제공됩니다.\n\n---\n\n**숙소 선택 시 고려할 점:**\n\n*   **여행 목적:** 휴양, 관광, 쇼핑, 먹거리, 비즈니스 등\n*   **예산:** 럭셔리, 중급, 가성비\n*   **동반자:** 가족 (아이 포함), 커플, 친구, 혼자\n*   **교통:** 대중교통 이용 여부, KTX 이용 여부\n*   **선호하는 분위기:** 조용함, 활기참, 로맨틱, 전통적\n\n어떤 숙소를 찾으시는지 조금 더 구체적으로 알려주시면, 더 자세히 추천해 드릴 수 있습니다!'
]
weather_results = [
    '부산 2026-02-07 날씨 예보:\n\n요약\n- 최고기온: 2.24°C\n- 최저기온: -6.01°C\n- 총 강수량: 0mm\n- 총 적설량: 0.0cm\n\n[2026-02-07 00:00:00]\n- 온도: -0.44°C (체감 -4.53°C)\n- 습도: 22%\n- 날씨: 온흐림\n- 풍속: 3.6 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 100%\n\n[2026-02-07 03:00:00]\n- 온도: 2.24°C (체감 -2.35°C)\n- 습도: 18%\n- 날씨: 튼구름\n- 풍속: 5.46 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 56%\n\n[2026-02-07 06:00:00]\n- 온도: 2.17°C (체감 -3.41°C)\n- 습도: 22%\n- 날씨: 구름조금\n- 풍속: 7.73 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 28%\n\n[2026-02-07 09:00:00]\n- 온도: -0.99°C (체감 -7.7°C)\n- 습도: 29%\n- 날씨: 튼구름\n- 풍속: 8.21 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 64%\n\n[2026-02-07 12:00:00]\n- 온도: -3.56°C (체감 -10.56°C)\n- 습도: 41%\n- 날씨: 튼구름\n- 풍속: 7.96 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 73%\n\n[2026-02-07 15:00:00]\n- 온도: -4.4°C (체감 -11.4°C)\n- 습도: 46%\n- 날씨: 온흐림\n- 풍속: 7.36 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 94%\n\n[2026-02-07 18:00:00]\n- 온도: -5.38°C (체감 -12.38°C)\n- 습도: 48%\n- 날씨: 구름조금\n- 풍속: 6.99 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 47%\n\n[2026-02-07 21:00:00]\n- 온도: -6.01°C (체감 -13.01°C)\n- 습도: 47%\n- 날씨: 맑음\n- 풍속: 7.42 m/s\n- 강수량: 0mm\n- 적설량: 0.0cm\n- 구름양: 0%\n\n'
]


# ================================= 변경사항들 =================================
question = "부산의 내일 강수량과 바다마루 전복죽이라는 음식점의 운영시간과 부산의 숙박을 추천해줘" 

# 3. chatbot 용
context_text = "\n\n".join([d["text"] for d in documents])
web_text = "\n".join(web_results)
weather_text = "\n".join(weather_results)
summary = "사용자는 부산의 맛집 추천을 요청했고, 모델은 해운대의 돼지 국밥을 추천하며 그 음식의 따뜻함과 든든함을 강조했다."

# 4. validator 용
bot_response = "부산에서 또 다른 맛집으로 해운대 돼지 국밥 추천해! 국밥 한 그릇이면 따뜻하고 든든하니 완전 좋음 ㅎㅎ"


prompt = ChatPromptTemplate.from_messages([
    ("system", Validator.system), 
    ("human", Validator.user),
])

formatted = prompt.format_messages(
    # 1. router
    # current_time=current_time,
    # days=days,
    # question=question,
    
    # 2. validator
    question=question,
    documents=documents,
    web_results=web_results,
    weather_results=weather_results,
    
    # 3. chatbot
    # question=question,
    # context_text=context_text,
    # web_text=web_text,
    # weather_text=weather_text,
    # summary=summary,
    
    # 4. summarize
    # question=question,
    # bot_response=bot_response
)
print("system: ")
print(formatted[-2].content)
print("human: ")
print(formatted[-1].content)

# 2. 
# prompt = system_prompt + user_prompt
# print(formatted)

# router, validator 용 -> structured_llm / 다른거 -> llm
chain = prompt | structured_llm

decision = chain.invoke({
    # 1. router
    # "current_time": current_time,
    # "days": days,  
    # "question": question
    
    # 2. validator
    "question":question,
    "documents":documents,
    "web_results":web_results,
    "weather_results":weather_results,
    
    # 3. chatbot
    # "question":question,
    # "context_text":context_text,
    # "web_text":web_text,
    # "weather_text":weather_text,
    # "summary":summary,
    
    # 4. summarize
    # "question":question,
    # "bot_response":bot_response
    
})


print("\n==================모델의 응답==================")
print(decision)
# print(decision.content)

system: 

당신은 검색 결과가 사용자의 질문을 해결하기에 충분한지 판단하는 **Sufficiency Validator(충족 여부 판단기)**입니다.
사용자 질문과 제공된 검색 결과(rag, web, weather)를 비교하여 다음 기준에 따라 판단하세요.

**판단 프로세스 (반드시 순서대로 적용)**

**1단계: 고유 명사 완전 일치 확인 (Specific Entity Rule)**
* **원칙:** 사용자가 특정 상호명, 브랜드명, 장소명(예: '바다마루전복죽')을 명시했다면, 검색 결과에 **정확히 그 이름**을 가진 대상의 정보가 있어야 합니다.
* **불충분 기준:** 이름이 비슷한 다른 가게, 같은 메뉴를 파는 다른 식당, 위치만 같은 다른 장소 정보는 **모두 불충분**으로 간주합니다.
    * *(예시) 질문: "바다마루전복죽 영업시간" / 결과: "원조전복죽 정보..." -> **불충분 (정보 없음)** *

**2단계: 일반 추천 및 정보 확인 (General Info Rule)**
* **원칙:** 사용자가 특정 대상을 콕 집지 않고 "추천해줘", "날씨 어때" 등 일반적인 정보를 요청한 경우에 적용합니다.
* **완벽주의 금지:** 정보가 1개라도 확실하게 있다면 "충분하다"고 판단하세요.

**3단계: 검색 채널 결정 (RAG 실패 시 Web 전환)**
* **상황:** 당신에게 `rag 결과`가 제공되었다는 것은 이미 내부 DB 검색을 1회 수행했다는 뜻입니다.
* **규칙:** 1, 2단계에서 찾는 정보가 `rag 결과`에 없다면, **절대로 `rag_queries`를 다시 생성하지 마세요.**
* **행동:** 부족한 정보는 반드시 **web_queries**에 추가하여 외부 웹 검색으로 해결해야 합니다.

**출력 형식 (JSON)**
{
    "rag_queries": ["쿼리", "쿼리"] 또는 null, 
    "web_queries": ["쿼리", "쿼리"] 또는 null,
    "weather_querie

In [8]:
# # # add system prompt
# # question = "부산의 내일 강수량과 바다마루 전복죽이라는 음식점의 운영시간과 부산의 숙박을 추천해줘"
# # question = "부산의 숙박을 추천해줘"
# question = "부산 해운대의 깨끗한 호텔이랑 내일 부산 날씨, 그리고 서울에서 부산까지 가는 srt 기차표의 제일 이른 시간대를 알려줘"

# messages = [
#             SystemMessage(content=gpt_prompt),
#             HumanMessage(content=question),
#         ]
# decision = structured_llm.invoke(messages)
# rag = decision.rag_queries
# web = decision.web_queries
# weather = decision.weather_queries
# direct = decision.direct
# reason = decision.route_reason
# print("RAG!!!!")
# print(rag)

# print("web!!!!")
# print(web)

# print("weather!!!!")
# print(weather)

# print("direct!!!!")
# print(direct)

# print("reason!!!!")
# print(reason)


In [9]:
while True:
    full_response = ""
    user_input = input("User: ")
    if user_input.lower() in ["q", "quit"]:
        print("종료합니다.")
        full_response += "종료합니다."
        break
    
    inputs = {"messages": [HumanMessage(content=user_input)]}
    
    result = app.invoke(inputs, config=config)
    
    ai_msg = result["final_answer"]
    route = result.get("route", "알 수 없음")
    summary = result.get("summary", "")

    print(f"AI: {ai_msg}")
    full_response += f"AI: {ai_msg}"
    print(f" └─ [Debug] 경로: {route}")
    full_response += f" └─ [Debug] 경로: {route}"
    
    if summary:
        print(f"   └─ [Debug] 📝 요약 발생: {summary}")
        full_response += f"   └─ [Debug] 📝 요약 발생: {summary}"
    
    print("-" * 40)

[Router]: 경로 탐색 중...
[Router][PROMPT] system: 

오늘 날짜는 다음과 같습니다: 2026년 02월 06일 금요일
일주일은 다음 요일을 순서대로 포함합니다.: ['월', '화', '수', '목', '금', '토', '일']
당신은 사용자 질문을 [rag, web, weather, direct]로 분류하는 Router입니다.
아래 규칙을 엄격히 준수하여 JSON을 생성하세요.

**★핵심 제약 사항 (위반 시 오답 처리):**
1. **개수 제한**: 각 카테고리(주제) 당 **가장 정확한 '단 하나(1개)'의 쿼리**만 생성하세요.
2. **중복 금지**: 같은 의미의 질문을 여러 번 쓰지 마세요. (유의어 나열 금지)
3. **날씨 포맷**: weather_queries는 반드시 숫자 형식이어야 합니다.

**분류 규칙:**
1. **rag**: 장소, 숙박, 맛집, 명소 정보, 액티비티, 레저 -> 리스트에 **핵심 키워드 1개**만.
2. **web**: 실시간 정보(교통편 시간표, 티켓 예매, 뉴스) -> 리스트에 **핵심 문장 1개**만.
3. **weather**: 날씨/기온 -> (내일=1, 오늘=0, 이틀후=2) 형식.
4. **direct**: 인사, 농담.

**출력 예시 (반드시 이 형태를 따를 것):**
User: "내일 부산 날씨랑 해운대 깨끗한 호텔 추천해주고 서울 가는 기차표 제일 빠른거 알려줘"
Output:
{
    "rag_queries": ["해운대 깨끗한 호텔"], 
    "web_queries": ["서울행 부산 출발 기차표 최단시간"],
    "weather_queries": [1],
    "direct": null,
}

[Router][PROMPT] human: 

유저의 질문: 부산의 내일 강수량과 바다마루 전복죽이라는 음식점의 운영시간과 부산의 숙박을 추천해줘

[Router] model is thinking...
[Router] Router analyzed the ques